# 05-03. Transport Matrix of PSLNMD  
# PSLNMD の輸送行列を作る

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

## 今日の目的 / Goals

05-01 では、4-box から Toggweiler 型 6-box へ進む理由を学びました。  
In 05-01, we learned why we move from the 4-box model to the Toggweiler-style six-box model.

05-02 では、PSLNMD を地理と深さのイメージで理解しました。  
In 05-02, we understood PSLNMD in terms of geography and depth.

この Notebook では、PSLNMD の **輸送行列** を作ります。  
In this notebook, we build the **transport matrix** for PSLNMD.

> **箱のつながりを、Python で計算できる形にする。**  
> **Convert box connectivity into a form that Python can compute.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: なぜ輸送行列が必要か / Why do we need a transport matrix?

ボックスモデルでは、各 box の濃度変化は、

```text
入ってくる量 - 出ていく量
incoming amount - outgoing amount
```

で決まります。

箱が少ないうちは式を手で書けますが、6-box では流れの数が増えます。  
With a few boxes, equations can be written by hand, but in a six-box model, the number of flows increases.

そのため、

```text
どの箱からどの箱へ流れるか
どの流量で流れるか
質量保存しているか
```

を整理する必要があります。  
We need to organize where water flows, how large transports are, and whether mass is conserved.

そのために使うのが **輸送行列** です。  
This is why we use a **transport matrix**.

## 2. PSLNMD の箱と体積 / Boxes and volumes

今回使う箱は、

```text
P, S, L, N, M, D
```

です。

ここでは教育用の仮の体積を使います。  
Here we use provisional teaching values.

In [ ]:
boxes = ["P", "S", "L", "N", "M", "D"]

VOCN_TOTAL = 1.292e18
VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())
volumes = np.array([VOLUME[b] for b in boxes])

pd.DataFrame({
    "Box": boxes,
    "Japanese": ["極域表層", "亜寒帯域表層", "低緯度表層", "北大西洋表層", "中層", "深層"],
    "English": ["Polar surface", "Subpolar surface", "Low-latitude surface",
                "North Atlantic surface", "Mid-depth", "Deep ocean"],
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in boxes],
})

## 3. Prediction: どんな流れが必要か / What flows do we need?

PSLNMD の最小循環として、まず次の経路を考えます。  
As a minimal PSLNMD circulation, we first consider:

```text
N → M → D → S → P → L → N
```

これは、

```text
N: 北大西洋で沈み込む / sinking in the North Atlantic
M: 中層を通る / mid-depth pathway
D: 深層へ入る / deep ocean
S: 南大洋で表層へ戻る / return to surface in the Southern Ocean
P, L: 表層を通って N へ戻る / surface return pathway to N
```

というイメージです。

In [ ]:
Q = 2.0e7  # m3/s
pathway = [("N", "M"), ("M", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]

pd.DataFrame(pathway, columns=["From", "To"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pos = {
    "P": (0.12, 0.70),
    "S": (0.36, 0.70),
    "L": (0.60, 0.70),
    "N": (0.84, 0.70),
    "M": (0.50, 0.43),
    "D": (0.50, 0.18),
}

for b, (x, y) in pos.items():
    ax.text(x, y, b, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.55", fc="white", ec="black"))

for a, b in pathway:
    ax.annotate("", xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle="->"))

ax.set_xlim(0, 1)
ax.set_ylim(0, 0.95)
ax.axis("off")
ax.set_title("Minimal transport pathway: N → M → D → S → P → L → N")
plt.show()

## 4. 質量保存を確認する / Check mass conservation

閉じた循環では、各 box の流入と流出が等しくなります。  
In a closed circulation, inflow and outflow are equal for each box.

In [ ]:
rows = []
for b in boxes:
    inflow = sum(Q for a, bb in pathway if bb == b)
    outflow = sum(Q for aa, b2 in pathway if aa == b)
    rows.append({
        "Box": b,
        "Inflow [m3/s]": inflow,
        "Outflow [m3/s]": outflow,
        "Inflow - Outflow": inflow - outflow,
    })

pd.DataFrame(rows)

### Mini exercise 1 / ミニ演習 1

すべての box で `Inflow - Outflow` は 0 ですか。  
Is `Inflow - Outflow` zero for all boxes?

もし 0 でなければ、保存トレーサーの総量が変化してしまう可能性があります。  
If not, the total amount of a conserved tracer may change.

## 5. 輸送行列の考え方 / Concept of transport matrix

濃度ベクトルを

```text
C = [C_P, C_S, C_L, C_N, C_M, C_D]
```

とします。

輸送行列 \(F\) を使うと、

```text
tendency = F @ C
```

と書けます。

ここで、

```text
F[target, source] > 0
```

は、source から target へ入る流れを表します。  
`F[target, source] > 0` means flow from source into target.

対角成分は、その box から出ていく流れです。  
Diagonal terms represent outflow from that box.

In [ ]:
def build_flux_matrix(pathway, Q, boxes):
    n = len(boxes)
    idx = {b: i for i, b in enumerate(boxes)}
    F = np.zeros((n, n))

    for source, target in pathway:
        i = idx[target]
        j = idx[source]

        F[i, j] += Q   # target receives water from source
        F[j, j] -= Q   # source loses water to target

    return F

F = build_flux_matrix(pathway, Q, boxes)
idx = {b: i for i, b in enumerate(boxes)}

pd.DataFrame(F, index=boxes, columns=boxes)

### Mini exercise 2 / ミニ演習 2

`D → S` に対応する要素はどこですか。  
Which matrix elements correspond to `D → S`?

`D → S` は、

```text
S receives from D
D loses to S
```

です。

In [ ]:
print("F[S, D] =", F[idx["S"], idx["D"]])
print("F[D, D] =", F[idx["D"], idx["D"]])

## 6. 1 step の輸送 / One transport step

濃度 \(C\) の 1 step 更新は、

```text
C_new = C + (F @ C) * dt / V
```

です。  
Here \(V\) is the volume of each box.

In [ ]:
DT = 8.64e4

def one_step_transport(c, F, volumes, dt=DT):
    tendency = F @ c
    return c + tendency * dt / volumes

# initial tracer: dye in N
c0 = np.zeros(len(boxes))
c0[idx["N"]] = 1.0

c1 = one_step_transport(c0, F, volumes)

pd.DataFrame({
    "Box": boxes,
    "Initial": c0,
    "After one day": c1,
})

## 7. 保存トレーサーの総量 / Total amount of a conserved tracer

受動トレーサーなら、総量は保存されるはずです。  
For a passive tracer, the total amount should be conserved.

総量は、

```text
sum(C * V)
```

です。

In [ ]:
def total_amount(c):
    return np.sum(c * volumes)

print("Initial total:", total_amount(c0))
print("After one day total:", total_amount(c1))
print("Relative difference:", (total_amount(c1) - total_amount(c0)) / total_amount(c0))

## 8. 染料実験 / Dye experiment

N に入れた染料が、どの box に広がるかを見ます。  
We examine how dye released in N spreads to other boxes.

In [ ]:
def run_dye(source_box="N", years=1500, F=F):
    c = np.zeros(len(boxes))
    c[idx[source_box]] = 1.0
    hist = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365, "total": total_amount(c)}
            for i, b in enumerate(boxes):
                row[b] = c[i]
            hist.append(row)
        c = one_step_transport(c, F, volumes)

    return pd.DataFrame(hist)

dye_N = run_dye("N", years=1500)

plt.figure()
for b in boxes:
    plt.plot(dye_N["year"], dye_N[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in N")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
plt.plot(dye_N["year"], dye_N["total"])
plt.xlabel("Time [yr]")
plt.ylabel("Total dye amount")
plt.title("Conservation check")
plt.grid(True)
plt.show()

print("Initial total:", dye_N["total"].iloc[0])
print("Final total:", dye_N["total"].iloc[-1])
print("Relative difference:", (dye_N["total"].iloc[-1] - dye_N["total"].iloc[0]) / dye_N["total"].iloc[0])

## 9. 流量 Q を変える / Change transport strength Q

\(Q\) が大きいほど、水は速く動きます。  
Larger \(Q\) means faster movement of water.

### Prediction / 予想

Q を大きくすると、N に入れた染料は D に早く届くでしょうか。  
If Q is increased, will dye released in N reach D faster?

In [ ]:
Q_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]

plt.figure()
summary_Q = []

for q in Q_list:
    Fq = build_flux_matrix(pathway, q, boxes)
    h = run_dye("N", years=1500, F=Fq)
    plt.plot(h["year"], h["D"], label=f"Q={q:.1e}")
    summary_Q.append({
        "Q [m3/s]": q,
        "D at 500 yr": h.loc[h["year"] == 500, "D"].iloc[0],
        "D at 1500 yr": h["D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Dye in D")
plt.title("Effect of transport strength on dye arrival in D")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_Q)

## 10. 追加交換を入れる / Add extra exchanges

実際のモデルでは、1 本の閉じた循環だけでなく、追加の交換を入れることがあります。  
In real models, we often add exchange terms in addition to a single closed loop.

ここでは例として、

```text
S ↔ M
L ↔ M
```

を加えます。  
These represent simplified exchange between surface and mid-depth water.

In [ ]:
def build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges):
    F = build_flux_matrix(pathway, Q, boxes)
    idx = {b: i for i, b in enumerate(boxes)}

    for a, b, q in exchanges:
        ia = idx[a]
        ib = idx[b]

        # a -> b
        F[ib, ia] += q
        F[ia, ia] -= q

        # b -> a
        F[ia, ib] += q
        F[ib, ib] -= q

    return F

exchanges = [("S", "M", 0.3e7), ("L", "M", 0.2e7)]
F_ex = build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges)

pd.DataFrame(F_ex, index=boxes, columns=boxes)

In [ ]:
dye_N_ex = run_dye("N", years=1500, F=F_ex)

plt.figure()
for b in boxes:
    plt.plot(dye_N_ex["year"], dye_N_ex[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in N with S-M and L-M exchanges")
plt.legend()
plt.grid(True)
plt.show()

print("Initial total:", dye_N_ex["total"].iloc[0])
print("Final total:", dye_N_ex["total"].iloc[-1])
print("Relative difference:", (dye_N_ex["total"].iloc[-1] - dye_N_ex["total"].iloc[0]) / dye_N_ex["total"].iloc[0])

## 11. Debugging checklist / デバッグの確認リスト

輸送行列を作るときは、次を確認します。  
When building a transport matrix, check the following:

```text
1. すべての流れに source と target があるか
2. F[target, source] が正か
3. F[source, source] が負か
4. 各 box の流入と流出がつり合うか
5. 保存トレーサーの総量が保存されるか
```

```text
1. Does every flow have a source and target?
2. Is F[target, source] positive?
3. Is F[source, source] negative?
4. Are inflow and outflow balanced?
5. Is total passive tracer amount conserved?
```

In [ ]:
def check_transport_matrix(F, boxes):
    return pd.DataFrame({
        "Box": boxes,
        "Column sum": F.sum(axis=0),
        "Diagonal": np.diag(F),
    })

check_transport_matrix(F_ex, boxes)

## 12. 05-03 のまとめ / Summary of 05-03

この Notebook で学んだことは次です。  
What we learned:

1. 輸送行列は、box 間の流れをまとめて表す。  
   A transport matrix summarizes flows among boxes.

2. `F[target, source]` は、source から target への流れを表す。  
   `F[target, source]` represents flow from source to target.

3. 対角成分は、その box から出ていく流れを表す。  
   Diagonal terms represent outflow from each box.

4. 濃度の更新は `C + (F @ C) * dt / V` で書ける。  
   Concentration update can be written as `C + (F @ C) * dt / V`.

5. 保存トレーサーを使って、輸送行列をデバッグできる。  
   A passive tracer can be used to debug the transport matrix.

## Key message

> **輸送行列は、海洋ボックスモデルを「手書きの式」から「拡張可能なコード」へ変える道具である。**  
> **The transport matrix turns an ocean box model from hand-written equations into extensible code.**

## 13. 課題 / Exercises

### 課題 1 / Exercise 1

輸送行列を使う利点を説明せよ。  
Explain the advantage of using a transport matrix.

### 課題 2 / Exercise 2

`N → M` の流れは、輸送行列のどの成分に対応するか。  
Which elements of the transport matrix correspond to the flow `N → M`?

### 課題 3 / Exercise 3

保存トレーサーの総量を確認する理由を説明せよ。  
Explain why we check the total amount of a passive tracer.

### 課題 4 / Exercise 4

Q を大きくすると、染料の広がり方はどう変わるか。  
How does dye spreading change when Q is increased?

### 課題 5 / Exercise 5

追加交換を入れるとき、どのようなバグが起こりやすいか。  
What kinds of bugs are likely when adding extra exchanges?

## 想定解答 / Expected answers

### 課題 1

輸送行列を使うと、box 間の流れをまとめて扱える。箱が増えても式を整理しやすく、Python では `F @ C` として一括計算できる。  
A transport matrix allows flows among boxes to be handled together. Even when the number of boxes increases, equations remain organized and can be computed in Python as `F @ C`.

### 課題 2

`N → M` では、M が N から受け取るので `F[M, N]` が正になる。また N から流出するので `F[N, N]` に負の成分が入る。  
For `N → M`, M receives from N, so `F[M, N]` is positive. N loses water, so a negative term is added to `F[N, N]`.

### 課題 3

受動トレーサーは生成・消滅しないため、総量が保存されるはずである。保存されなければ、輸送行列の符号、体積で割る処理、追加交換の入れ方にバグがある可能性がある。  
A passive tracer has no source or sink, so its total amount should be conserved. If not, there may be a bug in matrix signs, volume division, or extra exchanges.

### 課題 4

Q を大きくすると、水の移動が速くなるため、染料はより早く M や D へ到達する。  
When Q is increased, water moves faster, so dye reaches M and D earlier.

### 課題 5

追加交換では、片方向だけを入れてしまう、対角成分を引き忘れる、符号を逆にする、体積で割る場所を間違える、などのバグが起こりやすい。  
Common bugs include adding only one direction of an exchange, forgetting to subtract from the diagonal, reversing signs, or applying volume division incorrectly.

## 14. 次へ / Next step

次の Notebook では、この輸送行列を使って、保存トレーサー実験をより詳しく行います。  
In the next notebook, we use this transport matrix for more detailed passive tracer experiments.

```text
05-04 Tracer Experiments in PSLNMD
```